<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/Playground/s6e5/3_add_Count_Encode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost – F1 Pit Stop Prediction
**Dataset:** Kaggle Playground Series S6E5
**Target:** `PitNextLap` (binary classification)

## Alur Notebook
1. Konfigurasi & Imports
2. Load Data
3. Preprocessing
4. Baseline Model
5. Hyperparameter Tuning (Optuna)
6. Evaluasi & Ekspor Submission

## 1. Konfigurasi

In [ ]:
# ── Ubah nilai di blok ini sesuai kebutuhan ──────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/DATASET/playground-series-s6e5'
TRAIN_PATH   = f'{DRIVE_BASE}/train.csv'
TEST_PATH    = f'{DRIVE_BASE}/test.csv'

TARGET_COL   = 'PitNextLap'
ID_COL       = 'id'

TEST_SIZE    = 0.2
RANDOM_STATE = 10

# Mapping manual untuk kolom Compound (kategori ban)
COMPOUND_MAPPING = {
    'WET': 0, 'SOFT': 1, 'MEDIUM': 2, 'INTERMEDIATE': 3, 'HARD': 4
}
# Numerikcal Encode
COUNT_ENCODE = ['Driver','Race']
# Kolom kategorikal yang di-encode dengan LabelEncoder
# LABEL_ENCODE_COLS = ['Race']

# Parameter baseline XGBoost
BASELINE_PARAMS = {
    'n_estimators'    : 200,
    'max_depth'       : 6,
    'learning_rate'   : 0.1,
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'random_state'    : RANDOM_STATE,
}

# Konfigurasi Optuna
N_TRIALS = 50
CV_FOLDS = 5
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
!pip install optuna -q

import pandas as pd
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.4 MB/s eta 0:00:00
Mounted at /content/drive


## 3. Load Data

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
df_train.head()

Train : (439140, 16)
Test  : (188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


## 4. Preprocessing

In [ ]:
print(type(COUNT_ENCODE))
print(COUNT_ENCODE)

<class 'list'>
['Driver', 'Race']


In [ ]:
def preprocess(train: pd.DataFrame, test: pd.DataFrame) -> tuple:
    """Encode kolom kategorikal dan set index. Return (train, test) yang sudah bersih."""
    train = train.copy()
    test  = test.copy()

    # Encode kolom Compound dengan mapping yang sudah ditentukan
    train['Compound'] = train['Compound'].map(COMPOUND_MAPPING)
    test['Compound']  = test['Compound'].map(COMPOUND_MAPPING)

    # Encode kolom kategorikal lain dengan LabelEncoder
    # for col in LABEL_ENCODE_COLS:
    #     le = LabelEncoder().fit(pd.concat([train[col], test[col]]))
    #     train[col] = le.transform(train[col])
    #     test[col]  = le.transform(test[col])

    for col in COUNT_ENCODE:
      count_map = train[col].value_counts().to_dict()
      train[col] = train[col].map(count_map)
      test[col] = test[col].map(count_map).fillna(0)

    # Set id sebagai index
    train.set_index(ID_COL, inplace=True)
    test.set_index(ID_COL, inplace=True)

    return train, test


df_train, df_test = preprocess(df_train, df_test)

X = df_train.drop(columns=[TARGET_COL])
y = df_train[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')
df_train.info()

X_train: (351312, 14)  |  X_val: (87828, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 439140 entries, 0 to 439139
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  439140 non-null  int64  
 1   Compound                439140 non-null  int64  
 2   Race                    439140 non-null  int64  
 3   Year                    439140 non-null  int64  
 4   PitStop                 439140 non-null  int64  
 5   LapNumber               439140 non-null  int64  
 6   Stint                   439140 non-null  int64  
 7   TyreLife                439140 non-null  float64
 8   Position                439140 non-null  int64  
 9   LapTime (s)             439140 non-null  float64
 10  LapTime_Delta           439140 non-null  float64
 11  Cumulative_Degradation  439140 non-null  float64
 12  RaceProgress            439140 non-null  float64
 13  Position_Change         439140 non

## 5. Baseline Model

In [ ]:
baseline = xgb.XGBClassifier(**BASELINE_PARAMS)
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_val)
print(f'Accuracy  : {accuracy_score(y_val, y_pred):.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

Accuracy  : 0.8946

Confusion Matrix:
[[66013  4410]
 [ 4844 12561]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.94      0.93     70423
         1.0       0.74      0.72      0.73     17405

    accuracy                           0.89     87828
   macro avg       0.84      0.83      0.83     87828
weighted avg       0.89      0.89      0.89     87828



## 6. Hyperparameter Tuning (Optuna)
Cari kombinasi parameter terbaik menggunakan cross-validation.

In [ ]:
def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state'    : RANDOM_STATE,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring='accuracy')
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS)

print(f'Best CV Accuracy : {study.best_value:.4f}')
print(f'Best Params      : {study.best_params}')

Best CV Accuracy : 0.8987
Best Params      : {'n_estimators': 404, 'max_depth': 10, 'learning_rate': 0.038473130847093404, 'subsample': 0.9057014781350448, 'colsample_bytree': 0.692425067096911}


## 7. Evaluasi & Ekspor Submission

In [ ]:
# Train ulang dengan parameter terbaik pada seluruh data training
best_model = xgb.XGBClassifier(**study.best_params, random_state=RANDOM_STATE)
best_model.fit(X, y)

# Evaluasi pada validation set
y_pred_best = best_model.predict(X_val)
print(f'Tuned Accuracy : {accuracy_score(y_val, y_pred_best):.4f}')
print(classification_report(y_val, y_pred_best))

# Buat submission
submission = pd.DataFrame({
    ID_COL    : df_test.index,
    TARGET_COL: best_model.predict(df_test),
})
submission.to_csv(f'{DRIVE_BASE}/submission_3_1.csv', index=False)
print('✓ submission.csv tersimpan.')
submission.head()

Tuned Accuracy : 0.9277
              precision    recall  f1-score   support

         0.0       0.95      0.96      0.96     70423
         1.0       0.82      0.81      0.82     17405

    accuracy                           0.93     87828
   macro avg       0.89      0.88      0.89     87828
weighted avg       0.93      0.93      0.93     87828

✓ submission.csv tersimpan.


,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,1
